# EpiShield-AI — Phase 1: Data Quality Assessment (complete)

This notebook profiles and validates the four source datasets **before** any cleaning:

| Dataset | File | Role |
|---|---|---|
| Regional | `regioni.csv` | daily COVID-19 records per region |
| National | `nazionale.csv` | daily COVID-19 records for Italy |
| Population | `popolazione.csv` | population by region, age band and sex |
| Notes | `note.csv` | reporting notes / explanations |

**Principles**
- Source DataFrames are **never modified**. All checks run on temporary copies.
- Nothing is imputed, deleted or corrected here. Every finding goes into a single **Issue Register** (Section 8) and row-level evidence is exported as CSV files for the Cleaning phase.
- Numbers in the conclusions are **computed**, not typed by hand.

**What is checked**
1. Structure, dtypes, duplicates, empty / whitespace-only values (all datasets)
2. Dates: validity, format, ordering, key uniqueness, completeness of region × day grid
3. Numeric validity: non-numeric text, negatives, decimals
4. Missingness: when each column is available, gaps after first availability, identical patterns, regional synchronisation
5. Arithmetic rules (hospitalised, active positives, total cases, test split) and linked mismatches
6. Time-series rules: daily Δ of cumulative columns vs daily columns, non-decreasing cumulative columns
7. **Cross-dataset** checks: sum of regions vs national, regional vs population keys / coordinates / NUTS
8. Population: age-band coverage, totals, coordinates, identifiers
9. Notes: dates, content, references from regional/national `note` codes, region mentions

**How to run:** open from inside the EpiShield-AI repository (needs `Src/Data Engineering/extract.py`), then *Run All*.

In [1]:
import sys
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:          # allows running as a plain script
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

# If the notebook is NOT inside the repo, set the project root here, e.g. Path(r"C:\Projects\EpiShield-AI")
PROJECT_DIR_OVERRIDE = None


def find_project_root():
    if PROJECT_DIR_OVERRIDE is not None:
        root = Path(PROJECT_DIR_OVERRIDE).resolve()
        if not (root / "Src" / "Data Engineering" / "extract.py").is_file():
            raise FileNotFoundError(f"extract.py not found under {root}")
        return root
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "Src" / "Data Engineering" / "extract.py").is_file():
            return path
    raise FileNotFoundError(
        "Project directory not found. Expected Src/Data Engineering/extract.py "
        "in the current folder or one of its parents (or set PROJECT_DIR_OVERRIDE)."
    )


PROJECT_DIR = find_project_root()
EXTRACT_DIR = str(PROJECT_DIR / "Src" / "Data Engineering")
if EXTRACT_DIR not in sys.path:
    sys.path.insert(0, EXTRACT_DIR)

OUTPUT_DIR = PROJECT_DIR / "Src" / "Data Engineering" / "outputs" / "data_quality"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Outputs will be written to:", OUTPUT_DIR)

Project directory: C:\Users\New Star Computer 25\EpiShield-AI
Outputs will be written to: C:\Users\New Star Computer 25\EpiShield-AI\Src\Data Engineering\outputs\data_quality


In [2]:
# ------------------------------------------------------------------
# Shared helpers
# ------------------------------------------------------------------
ISSUES = []        # the Issue Register (one dict per finding)
EXCEPTIONS = {}    # row-level evidence, exported as CSV
SAVED_FILES = []

SEVERITY_ORDER = {"High": 0, "Medium": 1, "Low": 2, "Info": 3}


def add_issue(dataset, area, check, severity, affected, detail, action):
    ISSUES.append({
        "dataset": dataset,
        "area": area,
        "check": check,
        "severity": severity,
        "affected": affected,
        "detail": detail,
        "suggested_cleaning_action": action,
    })


def save_csv(frame, filename, index=False):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=index, encoding="utf-8-sig")
    SAVED_FILES.append(path.name)


def missing_mask(series):
    """True for NaN/None, empty strings and whitespace-only strings."""
    text = series.astype("string")
    return (series.isna() | text.str.strip().eq("").fillna(False)).astype(bool)


def to_num(series):
    return pd.to_numeric(series.astype("string").str.strip(), errors="coerce")


def d(x):
    return "n/a" if pd.isna(x) else pd.Timestamp(x).strftime("%Y-%m-%d")


def pct(a, b):
    return round(a / b * 100, 2) if b else 0.0


def has_cols(df, cols):
    return all(c in df.columns for c in cols)


# Numeric columns shared by the regional and national files
NUM_COLS = [
    "ricoverati_con_sintomi", "terapia_intensiva", "totale_ospedalizzati",
    "isolamento_domiciliare", "totale_positivi", "variazione_totale_positivi",
    "nuovi_positivi", "dimessi_guariti", "deceduti",
    "casi_da_sospetto_diagnostico", "casi_da_screening", "totale_casi",
    "tamponi", "casi_testati", "ingressi_terapia_intensiva",
    "totale_positivi_test_molecolare", "totale_positivi_test_antigenico_rapido",
    "tamponi_test_molecolare", "tamponi_test_antigenico_rapido",
]
NEGATIVE_ALLOWED = {"variazione_totale_positivi"}   # a change can legitimately be negative
NOTE_COLS = ["note", "note_test", "note_casi"]

# Cumulative columns: they should never decrease from one day to the next
MONOTONIC_COLS = [
    "totale_casi", "deceduti", "dimessi_guariti", "tamponi", "casi_testati",
    "tamponi_test_molecolare", "tamponi_test_antigenico_rapido",
    "totale_positivi_test_molecolare", "totale_positivi_test_antigenico_rapido",
    "casi_da_sospetto_diagnostico", "casi_da_screening",
]

# target = sum(parts)
SUM_RULES = [
    ("Hospitalised", "totale_ospedalizzati", ["ricoverati_con_sintomi", "terapia_intensiva"], "Medium"),
    ("Active positives", "totale_positivi", ["totale_ospedalizzati", "isolamento_domiciliare"], "Medium"),
    ("Total cases", "totale_casi", ["totale_positivi", "dimessi_guariti", "deceduti"], "Medium"),
    ("Tests split", "tamponi", ["tamponi_test_molecolare", "tamponi_test_antigenico_rapido"], "Low"),
]

# cumulative column whose day-to-day change should equal a daily column
DELTA_RULES = [
    ("Daily new cases vs change in totale_casi", "totale_casi", "nuovi_positivi"),
    ("Change in active positives vs variazione_totale_positivi", "totale_positivi", "variazione_totale_positivi"),
]

print("Helpers ready.")

Helpers ready.


## 1. Load the source data and validate the extraction output

In [3]:
from extract import extract_data

data = extract_data()

EXPECTED_DATASETS = ["regioni", "nazionale", "popolazione", "note"]
missing_keys = [k for k in EXPECTED_DATASETS if k not in data]
if missing_keys:
    raise KeyError(f"extract_data() did not return: {missing_keys}")

df_regional = data["regioni"]
df_national = data["nazionale"]
df_population = data["popolazione"]
df_notes = data["note"]

DATASETS = {
    "Regional": df_regional,
    "National": df_national,
    "Population": df_population,
    "Notes": df_notes,
}

# Columns without which the corresponding section cannot run at all
REQUIRED = {
    "Regional": ["data", "codice_regione", "denominazione_regione"],
    "National": ["data"],
    "Population": ["codice_regione", "range_eta", "totale_genere_maschile",
                   "totale_genere_femminile", "totale_generale"],
    "Notes": ["data"],
}
# Columns we expect (their absence is reported as an issue but does not stop the run)
EXPECTED = {
    "Regional": ["stato", "lat", "long", "codice_nuts_1", "codice_nuts_2"] + NUM_COLS + NOTE_COLS,
    "National": ["stato"] + NUM_COLS + NOTE_COLS,
    "Population": ["denominazione_regione", "sigla_regione", "codice_nuts_1", "descrizione_nuts_1",
                   "codice_nuts_2", "latitudine_regione", "longitudine_regione"],
    "Notes": ["note"],
}

for name, frame in DATASETS.items():
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{name} is not a pandas DataFrame")
    if frame.empty:
        raise ValueError(f"{name} dataset is empty - check extract.py and the CSV paths")
    dup_cols = frame.columns[frame.columns.duplicated()].tolist()
    if dup_cols:
        raise ValueError(f"{name} has duplicate column names: {dup_cols}")
    absent_required = [c for c in REQUIRED[name] if c not in frame.columns]
    if absent_required:
        raise KeyError(f"{name}: required columns missing: {absent_required}")
    absent_expected = [c for c in EXPECTED[name] if c not in frame.columns]
    if absent_expected:
        add_issue(name, "Structure", "Expected columns missing from file", "High", len(absent_expected),
                  f"Missing columns: {absent_expected}",
                  "Check the source file / extract.py; downstream steps that use these columns must be adapted.")
        print(f"[WARNING] {name}: expected columns not found: {absent_expected}")

print("Extraction OK.\n")
for name, frame in DATASETS.items():
    print(f"{name:<11} rows={len(frame):>7,}  columns={frame.shape[1]:>3}  dtypes={dict(frame.dtypes.astype(str).value_counts())}")

regioni: 37401 rows, 30 columns
nazionale: 1781 rows, 24 columns
popolazione: 210 rows, 12 columns
note: 1617 rows, 2 columns
Extraction OK.

Regional    rows= 37,401  columns= 30  dtypes={'str': np.int64(30)}
National    rows=  1,781  columns= 24  dtypes={'str': np.int64(24)}
Population  rows=    210  columns= 12  dtypes={'str': np.int64(12)}
Notes       rows=  1,617  columns=  2  dtypes={'str': np.int64(2)}


## 2. Generic profile of every dataset

For each dataset: shape, dtypes, empty values (null, `""` and whitespace-only), full-row duplicates and a per-column profile. The per-column profiles are exported to CSV.

In [4]:
def column_profile(df):
    rows = []
    for c in df.columns:
        m = missing_mask(df[c])
        non_empty = df.loc[~m, c]
        blank_not_null = int((df[c].notna() & m).sum())     # "" or whitespace only
        rows.append({
            "column": c,
            "dtype": str(df[c].dtype),
            "missing_or_empty": int(m.sum()),
            "missing_pct": pct(int(m.sum()), len(df)),
            "blank_strings": blank_not_null,
            "unique_non_empty": int(non_empty.nunique()),
            "example": str(non_empty.iloc[0])[:60] if len(non_empty) else "",
        })
    return pd.DataFrame(rows)


summary_rows = []
PROFILES = {}
for name, frame in DATASETS.items():
    prof = column_profile(frame)
    PROFILES[name] = prof
    save_csv(prof, f"column_profile_{name.lower()}.csv")

    dup_rows = int(frame.duplicated().sum())
    missing_cells = int(prof["missing_or_empty"].sum())
    total_cells = len(frame) * frame.shape[1]
    summary_rows.append({
        "Dataset": name, "Rows": len(frame), "Columns": frame.shape[1],
        "Full duplicate rows": dup_rows,
        "Missing/empty cells": missing_cells,
        "Missing/empty %": pct(missing_cells, total_cells),
    })

    if dup_rows:
        add_issue(name, "Duplicates", "Fully duplicated rows", "High", dup_rows,
                  "Rows identical in every column.", "Drop exact duplicates after confirming they are not legitimate repeats.")
    empty_cols = prof.loc[prof["missing_pct"] == 100, "column"].tolist()
    if empty_cols:
        add_issue(name, "Missingness", "Columns that are 100% empty", "Medium", len(empty_cols),
                  f"Columns: {empty_cols}", "Drop from the cleaned model tables (keep in raw layer) or document why they are empty.")
    blanks = int(prof["blank_strings"].sum())
    if blanks:
        add_issue(name, "Missingness", "Blank / whitespace-only strings (not real nulls)", "Low", blanks,
                  "Cells containing '' or spaces; they are treated as missing by this notebook.",
                  "Convert blank strings to proper nulls during cleaning.")

quality_summary = pd.DataFrame(summary_rows)
display(quality_summary)
save_csv(quality_summary, "quality_summary.csv")

for name in DATASETS:
    print(f"\n--- {name}: columns with missing/empty values ---")
    p = PROFILES[name]
    display(p[p["missing_or_empty"] > 0].sort_values("missing_pct", ascending=False).reset_index(drop=True))

,Dataset,Rows,Columns,Full duplicate rows,Missing/empty cells,Missing/empty %
0,Regional,37401,30,0,213705,19.05
1,National,1781,24,0,10179,23.81
2,Population,210,12,0,0,0.00
3,Notes,1617,2,0,14,0.43



--- Regional: columns with missing/empty values ---


,column,dtype,missing_or_empty,missing_pct,blank_strings,unique_non_empty,example
0,note_test,str,37204,99.47,37204,142,Al momento non è possibile definire il dato de...
1,casi_da_screening,str,33999,90.90,33999,2205,171
2,casi_da_sospetto_diagnostico,str,33999,90.90,33999,2711,3113
3,note_casi,str,32767,87.61,32767,3873,c'è stato un errore nella comunicazione dei po...
4,note,str,27562,73.69,27562,5318,pd-IT-0002
5,tamponi_test_molecolare,str,6846,18.30,6846,28822,557650
6,tamponi_test_antigenico_rapido,str,6846,18.30,6846,29638,617
7,totale_positivi_test_molecolare,str,6846,18.30,6846,25195,38834
8,totale_positivi_test_antigenico_rapido,str,6846,18.30,6846,23866,0
9,codice_nuts_1,str,6846,18.30,6846,5,ITF



--- National: columns with missing/empty values ---


,column,dtype,missing_or_empty,missing_pct,blank_strings,unique_non_empty,example
0,note_casi,str,1781,100.00,1781,0,
1,note_test,str,1781,100.00,1781,0,
2,note,str,1737,97.53,1737,44,pd-IT-0003
3,casi_da_sospetto_diagnostico,str,1619,90.90,1619,160,223905
4,casi_da_screening,str,1619,90.90,1619,162,15801
5,tamponi_test_molecolare,str,326,18.30,326,1455,28617351
6,totale_positivi_test_molecolare,str,326,18.30,326,1454,2351466
7,tamponi_test_antigenico_rapido,str,326,18.30,326,1455,116859
8,totale_positivi_test_antigenico_rapido,str,326,18.30,326,1455,957
9,ingressi_terapia_intensiva,str,283,15.89,283,210,217



--- Population: columns with missing/empty values ---


,column,dtype,missing_or_empty,missing_pct,blank_strings,unique_non_empty,example



--- Notes: columns with missing/empty values ---


,column,dtype,missing_or_empty,missing_pct,blank_strings,unique_non_empty,example
0,note,str,14,0.87,14,1593,La Provincia di Brescia ha comunicato dati par...


## Reusable time-series checks (used for both Regional and National)

The same engine runs on both files so that the two are validated identically. It covers dates and keys, numeric validity, missingness over time, arithmetic rules and day-to-day rules.

In [5]:
def save_exceptions(name, frame):
    """Row-level evidence for the Cleaning phase (internal helper columns are tidied)."""
    if frame is None or len(frame) == 0:
        return
    out = frame.copy()
    out = out.drop(columns=[c for c in ["_grp"] if c in out.columns])
    out = out.rename(columns={"_day": "date_parsed", "_prev_day": "previous_date", "_gap": "gap_days"})
    EXCEPTIONS[name] = out.reset_index(drop=True)


def ts_prepare(df_raw, name, group_cols, key_cols):
    df = df_raw.copy()
    parsed = pd.to_datetime(df["data"], errors="coerce")
    if getattr(parsed.dt, "tz", None) is not None:
        parsed = parsed.dt.tz_localize(None)
    df["_day"] = parsed.dt.normalize()
    gcols = list(group_cols) if group_cols else ["_grp"]
    if not group_cols:
        df["_grp"] = 1
    num_cols = [c for c in NUM_COLS if c in df.columns]
    num = df.copy()
    for c in num_cols:
        num[c] = to_num(num[c])
    return {"name": name, "df": df, "num": num, "num_cols": num_cols,
            "gcols": gcols, "keys": list(key_cols)}


# ---------------------------------------------------------------- dates and keys
def ts_dates_and_keys(ctx):
    name, df, g = ctx["name"], ctx["df"], ctx["gcols"]
    print(f"=== {name}: dates and keys ===")

    invalid = int(df["_day"].isna().sum())
    valid = df.dropna(subset=["_day"])
    print("Rows:", len(df))
    print("Invalid / unparseable dates:", invalid)
    print("Date range:", d(valid["_day"].min()), "->", d(valid["_day"].max()))
    if invalid:
        add_issue(name, "Dates", "Unparseable dates", "High", invalid,
                  "Values in 'data' that pandas cannot parse.", "Fix or drop these rows; they cannot be placed on the timeline.")
        save_exceptions(f"{name.lower()}_invalid_dates", df.loc[df["_day"].isna(), ctx["keys"]])

    fmt = df["data"].astype("string").str.replace(r"\d", "9", regex=True).value_counts()
    print("\nDate text formats (digits shown as 9):")
    display(fmt.to_frame("rows"))
    if len(fmt) > 1:
        add_issue(name, "Dates", "Mixed date text formats", "Low", int(fmt.iloc[1:].sum()),
                  f"Formats found: {fmt.index.tolist()}", "Standardise to a single datetime format in cleaning.")

    hours = df["data"].astype("string").str.extract(r"[T ](\d{2}):\d{2}")[0].value_counts()
    print("\nHour of day inside the date text:")
    display(hours.to_frame("rows"))
    if len(hours) > 1:
        add_issue(name, "Dates", "Time-of-day differs between rows", "Low", int(hours.iloc[1:].sum()),
                  f"Hours found: {hours.to_dict()}. The same calendar day can carry different timestamps.",
                  "Convert to a plain date (drop the time) and join on the date, never on the raw text.")

    ordered = bool(df.groupby(g)["_day"].apply(lambda s: s.is_monotonic_increasing).all())
    print("\nChronologically ordered within each group:", ordered)
    if not ordered:
        add_issue(name, "Dates", "Rows not chronologically ordered", "Low", 0,
                  "Rows are not sorted by date within group.", "Sort by (group, date) before any lag/rolling feature.")

    dup = df.duplicated(subset=g + ["_day"], keep=False) & df["_day"].notna()
    print("Rows sharing the same (group, day) key:", int(dup.sum()))
    if dup.any():
        add_issue(name, "Keys", "Duplicate (group, date) keys", "High", int(dup.sum()),
                  "More than one row for the same group and day.", "Investigate; keep the correct row or aggregate.")
        save_exceptions(f"{name.lower()}_duplicate_keys", df.loc[dup, ctx["keys"] + g])
        display(df.loc[dup, ctx["keys"]].head(20))

    # completeness of the (group x day) grid
    all_days = pd.date_range(valid["_day"].min(), valid["_day"].max(), freq="D")
    grp_vals = valid[g[0]].drop_duplicates().tolist()
    full = pd.MultiIndex.from_product([grp_vals, all_days], names=[g[0], "_day"])
    present = pd.MultiIndex.from_frame(valid[[g[0], "_day"]].drop_duplicates())
    absent = full.difference(present)
    n_groups = len(grp_vals)
    print(f"\nGroups: {n_groups} | calendar days in range: {len(all_days)} | expected rows: {len(full):,} | actual rows: {len(valid):,}")
    print("Missing (group, day) combinations:", len(absent))
    if len(absent):
        absent_df = absent.to_frame(index=False).rename(columns={"_day": "date_missing"})
        display(absent_df.head(20))
        add_issue(name, "Completeness", "Missing (group, day) combinations", "High", len(absent),
                  f"Calendar days without a row; first {d(absent_df['date_missing'].min())}, last {d(absent_df['date_missing'].max())}.",
                  "Decide whether to reindex to a full calendar grid (and what to do with the gaps).")
        save_exceptions(f"{name.lower()}_missing_group_days", absent_df)
    if n_groups > 1:
        per_day = valid.groupby("_day").size().value_counts().sort_index()
        print("\nNumber of groups present per day (value -> number of days):")
        display(per_day.to_frame("days"))


# ---------------------------------------------------------------- numeric validity
def ts_numeric(ctx):
    name, df, keys = ctx["name"], ctx["df"], ctx["keys"]
    print(f"=== {name}: numeric validation ===")
    rows, frames = [], []
    for c in ctx["num_cols"]:
        m = missing_mask(df[c])
        raw = df.loc[~m, c].astype("string").str.strip()
        num = pd.to_numeric(raw, errors="coerce")
        invalid_idx = raw.index[num.isna().to_numpy()]
        neg_idx = num.index[(num < 0).fillna(False).to_numpy()]
        nonint_idx = num.index[((num.notna()) & ((num % 1) != 0)).to_numpy()]
        rows.append({
            "column": c, "available": int(len(raw)),
            "non_numeric_text": int(len(invalid_idx)),
            "negative": int(len(neg_idx)), "decimals": int(len(nonint_idx)),
            "min": num.min() if num.notna().any() else np.nan,
            "max": num.max() if num.notna().any() else np.nan,
            "non_numeric_examples": raw.loc[invalid_idx].unique()[:3].tolist(),
        })
        for idx, why in ((invalid_idx, "non-numeric text"), (neg_idx, "negative value"), (nonint_idx, "decimal value")):
            if len(idx):
                if why == "negative value" and c in NEGATIVE_ALLOWED:
                    continue
                frames.append(df.loc[idx, keys].assign(column=c, value=df.loc[idx, c], problem=why))
        if len(invalid_idx):
            add_issue(name, "Validity", f"Non-numeric values in {c}", "High", int(len(invalid_idx)),
                      f"Examples: {raw.loc[invalid_idx].unique()[:3].tolist()}", "Fix or convert to null before casting to numeric.")
        if len(neg_idx) and c not in NEGATIVE_ALLOWED:
            add_issue(name, "Validity", f"Negative values in {c}", "Medium", int(len(neg_idx)),
                      f"Minimum = {num.min()}. A count should not be negative.", "Investigate (usually a correction by the source); decide per row.")
        if len(nonint_idx):
            add_issue(name, "Validity", f"Decimal values in count column {c}", "Low", int(len(nonint_idx)),
                      "Counts should be integers.", "Round or investigate.")
    report = pd.DataFrame(rows)
    display(report)
    save_csv(report, f"numeric_validation_{name.lower()}.csv")
    if frames:
        save_exceptions(f"{name.lower()}_numeric_problems", pd.concat(frames, ignore_index=True))
    return report


# ---------------------------------------------------------------- missingness over time
def ts_missing(ctx):
    name, df, g = ctx["name"], ctx["df"], ctx["gcols"]
    day = df["_day"]
    print(f"=== {name}: missingness over time ===")
    n_groups = df[g[0]].nunique() if g != ["_grp"] else 1
    cols = [c for c in df.columns if not c.startswith("_") and missing_mask(df[c]).any()]
    rows, masks = [], {}
    for c in cols:
        m = missing_mask(df[c])
        masks[c] = m
        first_av, last_av = day[~m].min(), day[~m].max()
        if pd.notna(first_av):
            before = int((m & (day < first_av)).sum())
            inside = int((m & (day >= first_av) & (day <= last_av)).sum())
            after_last = int((m & (day > last_av)).sum())
        else:
            before, inside, after_last = int(m.sum()), 0, 0
        row = {
            "column": c, "missing": int(m.sum()), "missing_pct": pct(int(m.sum()), len(df)),
            "first_available": first_av, "last_available": last_av,
            "first_missing": day[m].min(), "last_missing": day[m].max(),
            "missing_before_first_available": before,
            "missing_inside_available_window": inside,
            "missing_after_last_available": after_last,
        }
        if n_groups > 1:
            frac = m.groupby(day).mean()
            row["days_partially_missing_across_groups"] = int(((frac > 0) & (frac < 1)).sum())
        rows.append(row)
    report = pd.DataFrame(rows)
    if report.empty:
        print("No missing values.")
        return report
    report = report.sort_values("missing_pct", ascending=False).reset_index(drop=True)
    display(report)
    save_csv(report, f"missingness_{name.lower()}.csv")

    # columns that are missing on exactly the same rows
    signatures = {}
    for c, m in masks.items():
        signatures.setdefault(m.to_numpy().tobytes(), []).append(c)
    same = [v for v in signatures.values() if len(v) > 1]
    print("\nColumns with an IDENTICAL missing pattern:")
    for grp in same:
        print("  ", grp)
    if not same:
        print("   none")

    for _, r in report.iterrows():
        is_note = r["column"] in NOTE_COLS
        if r["missing_pct"] >= 100:
            sev = "High"
            detail = "Column is empty in every row."
            action = "Drop from model tables or document why it is empty."
        else:
            sev = "Medium" if (r["missing_pct"] >= 1 or r["missing_inside_available_window"] > 0) else "Low"
            detail = (f"{r['missing']} missing ({r['missing_pct']}%). Available {d(r['first_available'])} -> {d(r['last_available'])}; "
                      f"{r['missing_before_first_available']} missing before the first value, "
                      f"{r['missing_inside_available_window']} missing INSIDE the available window, "
                      f"{r['missing_after_last_available']} missing after the last value.")
            if is_note:
                action = "Sparse comment column: empty means 'no comment'. Do not impute; use as a has_note flag or drop."
            else:
                action = "Do NOT fill with 0 (missing = not reported). "
                if r["missing_before_first_available"] > 0:
                    action += f"Not available before {d(r['first_available'])}. "
                if r["missing_after_last_available"] > 0:
                    action += f"Series stops after {d(r['last_available'])}. "
                if r["missing_inside_available_window"] > 0:
                    action += "Gaps inside the available window need a per-case decision. "
                action += "Use only the available period or add an availability flag."
        add_issue(name, "Missingness", f"Missing values in {r['column']}", sev, int(r["missing"]), detail, action)
    return report


# ---------------------------------------------------------------- arithmetic rules
def sum_rule(num, keys, gcols, label, target, parts):
    expected = num[parts].sum(axis=1, min_count=len(parts))
    diff = num[target] - expected
    valid = num[target].notna() & expected.notna()
    bad = valid & (diff != 0)
    cols = list(dict.fromkeys(keys + gcols + ["_day", target] + parts))
    exc = num.loc[bad, cols].copy()
    exc["expected"] = expected[bad]
    exc["difference"] = diff[bad]
    summary = {"rule": label, "formula": f"{target} = " + " + ".join(parts),
               "checked": int(valid.sum()), "matching": int((valid & ~bad).sum()), "mismatching": int(bad.sum())}
    summary["match_pct"] = pct(summary["matching"], summary["checked"])
    return summary, exc


def delta_rule(num, gcols, keys, cumul, flow):
    t = num.dropna(subset=["_day"]).sort_values(gcols + ["_day"]).copy()
    grp = t.groupby(gcols, sort=False)
    t["_prev_day"] = grp["_day"].shift(1)
    t["_gap"] = (t["_day"] - t["_prev_day"]).dt.days
    t["_delta"] = t[cumul] - grp[cumul].shift(1)
    comparable = t["_gap"].eq(1) & t["_delta"].notna() & t[flow].notna()
    t["difference"] = t["_delta"] - t[flow]
    bad = comparable & (t["difference"] != 0)
    cols = list(dict.fromkeys(keys + gcols + ["_day", "_prev_day", "_gap", cumul, "_delta", flow, "difference"]))
    exc = t.loc[bad, cols].rename(columns={"_delta": f"daily_change_in_{cumul}"})
    summary = {"comparable": int(comparable.sum()), "matching": int((comparable & ~bad).sum()),
               "mismatching": int(bad.sum()), "not_comparable": int(len(t) - comparable.sum())}
    summary["match_pct"] = pct(summary["matching"], summary["comparable"])
    return summary, exc, t


def monotonic_check(num, gcols, keys, cols):
    t = num.dropna(subset=["_day"]).sort_values(gcols + ["_day"])
    rows, frames = [], []
    for c in cols:
        if c not in t.columns:
            continue
        sub = t[t[c].notna()]
        change = sub.groupby(gcols, sort=False)[c].diff()
        dec = (change < 0).fillna(False)
        if int(dec.sum()):
            e = sub.loc[dec, list(dict.fromkeys(keys + gcols + ["_day", c]))].copy()
            e["change_vs_previous_value"] = change[dec]
            e.insert(0, "column", c)
            frames.append(e)
        rows.append({"column": c, "decreasing_steps": int(dec.sum()),
                     "largest_drop": float(change[dec].min()) if dec.any() else 0,
                     "first_decrease": sub.loc[dec, "_day"].min() if dec.any() else pd.NaT,
                     "last_decrease": sub.loc[dec, "_day"].max() if dec.any() else pd.NaT})
    return pd.DataFrame(rows), (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame())


def show_context(ctx, exc, cumul, flow, top=3, window=3):
    if exc is None or exc.empty:
        return
    num, g = ctx["num"], ctx["gcols"]
    top_rows = exc.assign(_abs=exc["difference"].abs()).nlargest(top, "_abs")
    for _, r in top_rows.iterrows():
        mask = num["_day"].between(r["_day"] - pd.Timedelta(days=window), r["_day"] + pd.Timedelta(days=window))
        for gc in g:
            mask &= (num[gc] == r[gc])
        show_cols = list(dict.fromkeys(ctx["keys"] + [cumul, flow]))
        label = " | ".join(str(r[k]) for k in ctx["keys"])
        print(f"\nContext around the mismatch: {label}  (difference = {r['difference']:+,.0f})")
        display(num.loc[mask].sort_values("_day")[show_cols])


def ts_rules(ctx):
    name, num, g, keys = ctx["name"], ctx["num"], ctx["gcols"], ctx["keys"]
    out = {}

    # ---- arithmetic rules
    print(f"=== {name}: arithmetic rules ===")
    summaries, exc_frames = [], []
    for label, target, parts, sev in SUM_RULES:
        if not has_cols(num, [target] + parts):
            continue
        s, e = sum_rule(num, keys, g, label, target, parts)
        summaries.append(s)
        if len(e):
            exc_frames.append(e.assign(rule=label))
            save_exceptions(f"{name.lower()}_rule_{label.split(' (')[0].lower().replace(' ', '_')}", e)
            worst = e.loc[e["difference"].abs().idxmax()]
            add_issue(name, "Consistency", f"{label}: {s['formula']}", sev, s["mismatching"],
                      f"{s['mismatching']} of {s['checked']} rows break the rule ({100 - s['match_pct']:.2f}%). "
                      f"Dates {d(e['_day'].min())} -> {d(e['_day'].max())}; largest |difference| = {abs(worst['difference']):,.0f} on {d(worst['_day'])}.",
                      "Keep raw values; decide which component is wrong (see linked mismatches) and whether to recompute the total.")
    rule_table = pd.DataFrame(summaries)
    display(rule_table)
    save_csv(rule_table, f"arithmetic_rules_{name.lower()}.csv")
    out["rule_table"] = rule_table

    if exc_frames:
        all_exc = pd.concat(exc_frames, ignore_index=True)
        linked = (all_exc.groupby(keys)["rule"]
                  .agg(rules=lambda s: " | ".join(sorted(set(s))), n_rules="nunique").reset_index())
        linked = linked[linked["n_rules"] >= 2]
        print("\nSame row breaking 2+ rules (likely ONE faulty source field, not several):")
        if linked.empty:
            print("   none")
        else:
            display(linked)
            add_issue(name, "Consistency", "Rows breaking several arithmetic rules at once", "Medium", len(linked),
                      "A single wrong component often breaks two rules (e.g. totale_ospedalizzati breaks both 'Hospitalised' and 'Active positives'). See exceptions file.",
                      "Treat these as one root cause when correcting.")
            save_exceptions(f"{name.lower()}_linked_rule_mismatches", linked)
        if len(all_exc) <= 60:
            print("\nAll arithmetic mismatches:")
            display(all_exc.drop(columns=[c for c in ["_grp"] if c in all_exc.columns]))

    # ---- day-to-day rules
    print(f"\n=== {name}: day-to-day rules (only for consecutive calendar days) ===")
    for label, cumul, flow in DELTA_RULES:
        if not has_cols(num, [cumul, flow]):
            continue
        s, e, t_full = delta_rule(num, g, keys, cumul, flow)
        print(f"\n{label}")
        print(f"   comparable: {s['comparable']:,} | matching: {s['matching']:,} | mismatching: {s['mismatching']:,} "
              f"| match: {s['match_pct']}% | not comparable (first row of group / date gap / missing): {s['not_comparable']:,}")
        out[label] = s
        if not e.empty:
            e2 = e.copy()
            e2["direction"] = np.where(e2["difference"] < 0, "negative", "positive")
            e2["year"] = e2["_day"].dt.year
            by_year = e2.groupby(["year", "direction"]).agg(
                mismatches=("difference", "size"), total_difference=("difference", "sum"),
                max_abs_difference=("difference", lambda x: x.abs().max())).reset_index()
            print("Mismatches by year and direction:")
            display(by_year)
            labels = ["0-1", "2-5", "6-100", "101-1000", ">1000"]
            bins = pd.cut(e2["difference"].abs(), [-1, 1, 5, 100, 1000, np.inf], labels=labels)
            buckets = bins.value_counts().reindex(labels).fillna(0).astype(int)
            print("Mismatches by size of |difference|:")
            display(buckets.to_frame("mismatches"))
            small = int(buckets[["0-1", "2-5"]].sum()); big = int(buckets[["101-1000", ">1000"]].sum())
            if g != ["_grp"]:
                by_grp = e2.groupby(g).size().sort_values(ascending=False).head(10).to_frame("mismatches")
                print("Groups with most mismatches (top 10):")
                display(by_grp)
            print("Largest absolute differences (top 10):")
            display(e2.assign(abs_difference=e2["difference"].abs()).nlargest(10, "abs_difference")
                    .drop(columns=[c for c in ["_grp", "_prev_day", "_gap"] if c in e2.columns]))
            show_context(ctx, e, cumul, flow, top=2)
            neg = int((e2["difference"] < 0).sum()); pos = int((e2["difference"] > 0).sum())
            worst = e2.loc[e2["difference"].abs().idxmax()]
            sev = "High" if big > 0 else "Medium"
            add_issue(name, "Consistency", label, sev, s["mismatching"],
                      f"{s['mismatching']} of {s['comparable']} comparable rows mismatch ({100 - s['match_pct']:.2f}%): {neg} negative, {pos} positive. "
                      f"{small} of them are within +/-5, {big} exceed 100. "
                      f"Largest |difference| = {abs(worst['difference']):,.0f} on {d(worst['_day'])}. Cause not established by this notebook "
                      "(check the source documentation about retroactive revisions / backfilled cases).",
                      f"Do not trust '{flow}' as the daily change of '{cumul}' without reconciling; decide which of the two to use as the target series.")
            save_exceptions(f"{name.lower()}_daily_vs_cumulative_{flow}", e)
        else:
            print("   no mismatches")

    print(f"\n=== {name}: cumulative columns must not decrease ===")
    mono_table, mono_exc = monotonic_check(num, g, keys, MONOTONIC_COLS)
    display(mono_table)
    save_csv(mono_table, f"monotonic_check_{name.lower()}.csv")
    if not mono_exc.empty:
        save_exceptions(f"{name.lower()}_decreasing_cumulative", mono_exc)
        for _, r in mono_table[mono_table["decreasing_steps"] > 0].iterrows():
            add_issue(name, "Consistency", f"Cumulative column {r['column']} decreases", "Medium", int(r["decreasing_steps"]),
                      f"{int(r['decreasing_steps'])} decreases (largest drop {r['largest_drop']:,.0f}); {d(r['first_decrease'])} -> {d(r['last_decrease'])}.",
                      "Usually a source correction. Do not take diff() of this column as daily counts without handling negatives.")
    out["monotonic"] = mono_table
    return out


print("Time-series engine ready.")

Time-series engine ready.


## 3. Regional dataset (`regioni.csv`)

In [6]:
REG_KEYS = ["data", "codice_regione", "denominazione_regione"]
REG = ts_prepare(df_regional, "Regional", ["codice_regione"], REG_KEYS)
ts_dates_and_keys(REG)

=== Regional: dates and keys ===
Rows: 37401
Invalid / unparseable dates: 0
Date range: 2020-02-24 -> 2025-01-08

Date text formats (digits shown as 9):


,rows
data,
9999-99-99T99:99:99,37401



Hour of day inside the date text:


,rows
0,
17,37170
18,231



Chronologically ordered within each group: True
Rows sharing the same (group, day) key: 0

Groups: 21 | calendar days in range: 1781 | expected rows: 37,401 | actual rows: 37,401
Missing (group, day) combinations: 0

Number of groups present per day (value -> number of days):


,days
21,1781


In [7]:
# Identifiers: one name per code and one code per name, no stray whitespace
reg_df = REG["df"]
pairs = reg_df[["codice_regione", "denominazione_regione"]].drop_duplicates()
names_per_code = pairs.groupby("codice_regione")["denominazione_regione"].nunique()
codes_per_name = pairs.groupby("denominazione_regione")["codice_regione"].nunique()
print("Distinct region codes:", reg_df["codice_regione"].nunique(), "| distinct names:", reg_df["denominazione_regione"].nunique())
print("Codes with more than one name:", int((names_per_code > 1).sum()))
print("Names with more than one code:", int((codes_per_name > 1).sum()))
blank_id = int((missing_mask(reg_df["codice_regione"]) | missing_mask(reg_df["denominazione_regione"])).sum())
untrimmed = int((reg_df["denominazione_regione"].astype("string") != reg_df["denominazione_regione"].astype("string").str.strip()).sum())
print("Rows with blank code/name:", blank_id, "| names with leading/trailing spaces:", untrimmed)
if (names_per_code > 1).any() or (codes_per_name > 1).any():
    add_issue("Regional", "Keys", "Region code/name not one-to-one", "High", int((names_per_code > 1).sum() + (codes_per_name > 1).sum()),
              "A code maps to several names or the reverse.", "Build a clean region dimension table before joining.")
if blank_id:
    add_issue("Regional", "Keys", "Blank region identifiers", "High", blank_id, "Empty code or name.", "Fix or drop.")
if untrimmed:
    add_issue("Regional", "Keys", "Region names with surrounding whitespace", "Low", untrimmed, "Leading/trailing spaces.", "Strip text columns in cleaning.")

reg_dim = pairs.sort_values("codice_regione").reset_index(drop=True)
rows_per_region = reg_df.groupby("codice_regione").size().rename("rows")
display(reg_dim.merge(rows_per_region, on="codice_regione"))

Distinct region codes: 21 | distinct names: 21
Codes with more than one name: 0
Names with more than one code: 0
Rows with blank code/name: 0 | names with leading/trailing spaces: 0


,codice_regione,denominazione_regione,rows
0,01,Piemonte,1781
1,02,Valle d'Aosta,1781
2,03,Lombardia,1781
3,05,Veneto,1781
4,06,Friuli Venezia Giulia,1781
5,07,Liguria,1781
6,08,Emilia-Romagna,1781
7,09,Toscana,1781
8,10,Umbria,1781
9,11,Marche,1781


In [8]:
# stato / lat / long
if "stato" in df_regional.columns:
    print("Values of 'stato':")
    display(df_regional["stato"].value_counts(dropna=False).to_frame("rows"))
    if df_regional["stato"].nunique(dropna=False) > 1:
        add_issue("Regional", "Validity", "'stato' has more than one value", "Low", df_regional["stato"].nunique(dropna=False),
                  "Expected a constant country code.", "Verify; drop the column if constant.")

if has_cols(df_regional, ["lat", "long"]):
    geo = reg_df[["codice_regione", "lat", "long"]].copy()
    geo["lat"], geo["long"] = to_num(geo["lat"]), to_num(geo["long"])
    bad_geo = int(geo["lat"].isna().sum() + geo["long"].isna().sum())
    per_region = geo.groupby("codice_regione").agg(lat_unique=("lat", "nunique"), long_unique=("long", "nunique"),
                                                   lat=("lat", "first"), long=("long", "first"))
    out_box = per_region[~per_region["lat"].between(35.4, 47.2) | ~per_region["long"].between(6.5, 18.6)]
    print("Non-numeric/missing coordinates:", bad_geo)
    print("Regions whose coordinates vary across rows:", int(((per_region["lat_unique"] > 1) | (per_region["long_unique"] > 1)).sum()))
    print("Regions outside the Italy bounding box (lat 35.4-47.2, long 6.5-18.6):", len(out_box))
    if bad_geo:
        add_issue("Regional", "Validity", "Non-numeric or missing lat/long", "Medium", bad_geo, "Coordinates cannot be parsed.", "Fix or fill from the region dimension.")
    if ((per_region["lat_unique"] > 1) | (per_region["long_unique"] > 1)).any():
        add_issue("Regional", "Validity", "Coordinates differ within the same region", "Medium",
                  int(((per_region["lat_unique"] > 1) | (per_region["long_unique"] > 1)).sum()), "A region has more than one lat/long.", "Keep one coordinate per region.")
    if len(out_box):
        add_issue("Regional", "Validity", "Region coordinates outside Italy", "Medium", len(out_box), f"Regions: {out_box.index.tolist()}", "Verify coordinates.")
    display(per_region)

# NUTS codes: one per region where present
for c in ["codice_nuts_1", "codice_nuts_2"]:
    if c in df_regional.columns:
        avail = reg_df.loc[~missing_mask(reg_df[c])]
        multi = int((avail.groupby("codice_regione")[c].nunique() > 1).sum())
        print(f"{c}: available in {len(avail):,} rows; regions with >1 distinct value: {multi}")
        if multi:
            add_issue("Regional", "Keys", f"{c} not unique per region", "Medium", multi, "A region has several NUTS values.", "Verify against the population file.")

Values of 'stato':


,rows
stato,
ITA,37401


Non-numeric/missing coordinates: 0
Regions whose coordinates vary across rows: 2
Regions outside the Italy bounding box (lat 35.4-47.2, long 6.5-18.6): 0


,lat_unique,long_unique,lat,long
codice_regione,,,,
01,1,1,45.073274,7.680687
02,1,1,45.737503,7.320149
03,1,1,45.466794,9.190347
05,1,1,45.434905,12.338452
06,1,1,45.649435,13.768136
07,1,1,44.411493,8.932699
08,1,2,44.494367,11.341721
09,1,1,43.769231,11.255889
10,1,1,43.106758,12.388247


codice_nuts_1: available in 30,555 rows; regions with >1 distinct value: 0
codice_nuts_2: available in 30,555 rows; regions with >1 distinct value: 0


In [9]:
reg_numeric_report = ts_numeric(REG)

=== Regional: numeric validation ===


,column,available,non_numeric_text,negative,decimals,min,max,non_numeric_examples
0,ricoverati_con_sintomi,37401,0,0,0,0,12077,[]
1,terapia_intensiva,37401,0,0,0,0,1381,[]
2,totale_ospedalizzati,37401,0,0,0,0,13328,[]
3,isolamento_domiciliare,37401,0,0,0,0,574548,[]
4,totale_positivi,37401,0,0,0,0,578257,[]
5,variazione_totale_positivi,37401,0,15337,0,-50797,47483,[]
6,nuovi_positivi,37401,0,14,0,-229,52693,[]
7,dimessi_guariti,37401,0,0,0,0,4342249,[]
8,deceduti,37401,0,0,0,0,48739,[]
9,casi_da_sospetto_diagnostico,3402,0,0,0,0,305002,[]


In [10]:
reg_missing_report = ts_missing(REG)

=== Regional: missingness over time ===


,column,missing,missing_pct,first_available,last_available,first_missing,last_missing,missing_before_first_available,missing_inside_available_window,missing_after_last_available,days_partially_missing_across_groups
0,note_test,37204,99.47,2020-12-04,2024-08-22,2020-02-24,2025-01-08,5964,28321,2919,119
1,casi_da_screening,33999,90.90,2020-06-25,2020-12-03,2020-02-24,2025-01-08,2562,0,31437,0
2,casi_da_sospetto_diagnostico,33999,90.90,2020-06-25,2020-12-03,2020-02-24,2025-01-08,2562,0,31437,0
3,note_casi,32767,87.61,2020-12-04,2025-01-08,2020-02-24,2025-01-08,5964,26803,0,1491
4,note,27562,73.69,2020-03-07,2025-01-08,2020-02-24,2025-01-08,252,27310,0,1694
5,tamponi_test_molecolare,6846,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,6846,0,0,0
6,tamponi_test_antigenico_rapido,6846,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,6846,0,0,0
7,totale_positivi_test_molecolare,6846,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,6846,0,0,0
8,totale_positivi_test_antigenico_rapido,6846,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,6846,0,0,0
9,codice_nuts_1,6846,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,6846,0,0,0



Columns with an IDENTICAL missing pattern:
   ['casi_da_sospetto_diagnostico', 'casi_da_screening']
   ['totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido', 'codice_nuts_1', 'codice_nuts_2']


In [11]:
reg_results = ts_rules(REG)

=== Regional: arithmetic rules ===


,rule,formula,checked,matching,mismatching,match_pct
0,Hospitalised,totale_ospedalizzati = ricoverati_con_sintomi ...,37401,37401,0,100.00
1,Active positives,totale_positivi = totale_ospedalizzati + isola...,37401,37400,1,100.00
2,Total cases,totale_casi = totale_positivi + dimessi_guarit...,37401,37386,15,99.96
3,Tests split,tamponi = tamponi_test_molecolare + tamponi_te...,30555,30555,0,100.00



Same row breaking 2+ rules (likely ONE faulty source field, not several):


,data,codice_regione,denominazione_regione,rules,n_rules
7,2021-11-07T17:00:00,05,Veneto,Active positives | Total cases,2



All arithmetic mismatches:


,data,codice_regione,denominazione_regione,_day,totale_positivi,totale_ospedalizzati,isolamento_domiciliare,expected,difference,rule,totale_casi,dimessi_guariti,deceduti
0,2021-11-07T17:00:00,05,Veneto,2021-11-07,12616,247,12375,12622,-6,Active positives,<NA>,<NA>,<NA>
1,2020-11-04T17:00:00,18,Calabria,2020-11-04,3893,<NA>,<NA>,6091,1,Total cases,6092,2074,124
2,2020-11-04T17:00:00,09,Toscana,2020-11-04,35464,<NA>,<NA>,52799,16,Total cases,52815,15890,1445
3,2021-02-03T17:00:00,08,Emilia-Romagna,2021-02-03,44402,<NA>,<NA>,221614,1,Total cases,221615,167556,9656
4,2021-06-23T17:00:00,06,Friuli Venezia Giulia,2021-06-23,4354,<NA>,<NA>,107487,15,Total cases,107502,99348,3785
5,2021-06-23T17:00:00,05,Veneto,2021-06-23,4896,<NA>,<NA>,425159,3,Total cases,425162,408655,11608
6,2021-08-01T17:00:00,05,Veneto,2021-08-01,12352,<NA>,<NA>,437063,3,Total cases,437066,413072,11639
7,2021-08-11T17:00:00,02,Valle d'Aosta,2021-08-11,129,<NA>,<NA>,11869,2,Total cases,11871,11267,473
8,2021-11-07T17:00:00,05,Veneto,2021-11-07,12616,<NA>,<NA>,484940,6,Total cases,484946,460476,11848
9,2022-01-11T17:00:00,15,Campania,2022-01-11,186220,<NA>,<NA>,732377,227,Total cases,732604,537536,8621


... (10 outputs left out)


Context around the mismatch: 2022-01-25T17:00:00 | 05 | Veneto  (difference = +24,568)


,data,codice_regione,denominazione_regione,totale_casi,nuovi_positivi
14678,2022-01-22T17:00:00,05,Veneto,993986,18773
14699,2022-01-23T17:00:00,05,Veneto,1008962,14976
14720,2022-01-24T17:00:00,05,Veneto,1015150,6188
14741,2022-01-25T17:00:00,05,Veneto,1064030,24312
14762,2022-01-26T17:00:00,05,Veneto,1083850,19820
14783,2022-01-27T17:00:00,05,Veneto,1102848,18998
14804,2022-01-28T17:00:00,05,Veneto,1119366,16518



Change in active positives vs variazione_totale_positivi
   comparable: 37,380 | matching: 37,380 | mismatching: 0 | match: 100.0% | not comparable (first row of group / date gap / missing): 21
   no mismatches

=== Regional: cumulative columns must not decrease ===


,column,decreasing_steps,largest_drop,first_decrease,last_decrease
0,totale_casi,48,-7558.0,2020-02-27,2025-01-03
1,deceduti,46,-100.0,2020-03-23,2024-12-22
2,dimessi_guariti,256,-17634.0,2020-03-12,2025-01-08
3,tamponi,18,-27380997.0,2020-02-26,2024-11-05
4,casi_testati,31,-25199916.0,2020-04-20,2024-12-20
5,tamponi_test_molecolare,11,-13905.0,2021-01-25,2024-08-22
6,tamponi_test_antigenico_rapido,10,-27381683.0,2021-03-26,2024-11-05
7,totale_positivi_test_molecolare,472,-569303.0,2021-01-21,2025-01-08
8,totale_positivi_test_antigenico_rapido,254,-569531.0,2021-01-17,2025-01-03
9,casi_da_sospetto_diagnostico,226,-305002.0,2020-06-26,2020-12-03


In [12]:
# Notes-reference columns in the regional file
CODE_PATTERN = r"[A-Za-z]+-[A-Za-z]{2}-\d+"


def note_tokens(series):
    vals = series[~missing_mask(series)].astype("string").str.strip()
    toks = vals.str.split(";").explode().str.strip()
    return toks[toks.notna() & toks.ne("")]


def profile_note_columns(df, name):
    codes = set()
    for c in NOTE_COLS:
        if c not in df.columns:
            continue
        m = missing_mask(df[c])
        toks = note_tokens(df[c])
        is_code = toks.str.fullmatch(CODE_PATTERN).fillna(False).astype(bool)
        multi = int(df.loc[~m, c].astype("string").str.contains(";", regex=False).sum())
        print(f"\n[{name}] {c}: non-empty rows={int((~m).sum()):,} | unique tokens={toks.nunique()} | rows with several ';' codes={multi} "
              f"| code-like tokens={int(is_code.sum())} | other text tokens={int((~is_code).sum())}")
        if len(toks):
            print("   prefixes:", toks[is_code].str.extract(r"^([A-Za-z]+)-")[0].value_counts().to_dict())
            if (~is_code).any():
                print("   free-text examples:", toks[~is_code].unique()[:3].tolist())
                add_issue(name, "Notes", f"Free text (not a code) in '{c}'", "Low", int((~is_code).sum()),
                          "Column mixes reference codes and free text.", "Separate code references from comments in cleaning.")
        if c == "note":
            codes = set(toks[is_code].tolist())
    return codes


reg_note_codes = profile_note_columns(df_regional, "Regional")
print("\nDistinct note codes referenced by the regional file:", len(reg_note_codes))


[Regional] note: non-empty rows=9,839 | unique tokens=5277 | rows with several ';' codes=103 | code-like tokens=54 | other text tokens=9934
   prefixes: {'dc': 36, 'pd': 11, 'nd': 7}
   free-text examples: ['n.3 decessi precedentemente segnalati, da una revisione non risultano classificabili come COVID-19 positivi', 'NEI CASI POSITIVI COMPRESI N. 2 PAZIENTI IN ISOLAMENTO DOMICILIARE PROVENIENTI DA ALTRA REGIONE', 'In seguito a verifica sono stati eliminati 4 casi da Ferrara in quanto inserimento duplicato dello stesso paziente con dati anagrafici non corretti. Il numero totale di ieri (25/06) pertanto è di 28347 e non 28351.']

[Regional] note_test: non-empty rows=197 | unique tokens=144 | rows with several ';' codes=42 | code-like tokens=0 | other text tokens=240
   prefixes: {}
   free-text examples: ['Al momento non è possibile definire il dato dei tamponi antigenici.', 'i 81323 test antigenici rapidi sono riferiti a partire dal 02/11/2020', 'in Regione Liguria la positività al tes

## 4. National dataset (`nazionale.csv`)

In [13]:
NAT = ts_prepare(df_national, "National", [], ["data"])
ts_dates_and_keys(NAT)

if "stato" in df_national.columns:
    print("\nValues of 'stato':")
    display(df_national["stato"].value_counts(dropna=False).to_frame("rows"))

=== National: dates and keys ===
Rows: 1781
Invalid / unparseable dates: 0
Date range: 2020-02-24 -> 2025-01-08

Date text formats (digits shown as 9):


,rows
data,
9999-99-99T99:99:99,1781



Hour of day inside the date text:


,rows
0,
17,1765
18,16



Chronologically ordered within each group: True
Rows sharing the same (group, day) key: 0

Groups: 1 | calendar days in range: 1781 | expected rows: 1,781 | actual rows: 1,781
Missing (group, day) combinations: 0

Values of 'stato':


,rows
stato,
ITA,1781


In [14]:
nat_numeric_report = ts_numeric(NAT)

=== National: numeric validation ===


,column,available,non_numeric_text,negative,decimals,min,max,non_numeric_examples
0,ricoverati_con_sintomi,1781,0,0,0,101,34697,[]
1,terapia_intensiva,1781,0,0,0,17,4068,[]
2,totale_ospedalizzati,1781,0,0,0,127,38507,[]
3,isolamento_domiciliare,1781,0,0,0,94,2713594,[]
4,totale_positivi,1781,0,0,0,221,2734906,[]
5,variazione_totale_positivi,1781,0,916,0,-116092,172462,[]
6,nuovi_positivi,1781,0,0,0,30,220532,[]
7,dimessi_guariti,1781,0,0,0,1,26562666,[]
8,deceduti,1781,0,0,0,7,198683,[]
9,casi_da_sospetto_diagnostico,162,0,0,0,0,988470,[]


In [15]:
nat_missing_report = ts_missing(NAT)

=== National: missingness over time ===


,column,missing,missing_pct,first_available,last_available,first_missing,last_missing,missing_before_first_available,missing_inside_available_window,missing_after_last_available
0,note_casi,1781,100.00,NaT,NaT,2020-02-24,2025-01-08,1781,0,0
1,note_test,1781,100.00,NaT,NaT,2020-02-24,2025-01-08,1781,0,0
2,note,1737,97.53,2020-03-07,2020-06-24,2020-02-24,2025-01-08,12,66,1659
3,casi_da_sospetto_diagnostico,1619,90.90,2020-06-25,2020-12-03,2020-02-24,2025-01-08,122,0,1497
4,casi_da_screening,1619,90.90,2020-06-25,2020-12-03,2020-02-24,2025-01-08,122,0,1497
5,tamponi_test_molecolare,326,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,326,0,0
6,totale_positivi_test_molecolare,326,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,326,0,0
7,tamponi_test_antigenico_rapido,326,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,326,0,0
8,totale_positivi_test_antigenico_rapido,326,18.30,2021-01-15,2025-01-08,2020-02-24,2021-01-14,326,0,0
9,ingressi_terapia_intensiva,283,15.89,2020-12-03,2025-01-08,2020-02-24,2020-12-02,283,0,0



Columns with an IDENTICAL missing pattern:
   ['casi_da_sospetto_diagnostico', 'casi_da_screening']
   ['note_test', 'note_casi']
   ['totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido']


In [16]:
nat_results = ts_rules(NAT)

=== National: arithmetic rules ===


,rule,formula,checked,matching,mismatching,match_pct
0,Hospitalised,totale_ospedalizzati = ricoverati_con_sintomi ...,1781,1780,1,99.94
1,Active positives,totale_positivi = totale_ospedalizzati + isola...,1781,1779,2,99.89
2,Total cases,totale_casi = totale_positivi + dimessi_guarit...,1781,1768,13,99.27
3,Tests split,tamponi = tamponi_test_molecolare + tamponi_te...,1455,1455,0,100.00



Same row breaking 2+ rules (likely ONE faulty source field, not several):


,data,rules,n_rules
0,2020-02-25T18:00:00,Active positives | Hospitalised,2
6,2021-11-07T17:00:00,Active positives | Total cases,2



All arithmetic mismatches:


,data,_day,totale_ospedalizzati,ricoverati_con_sintomi,terapia_intensiva,expected,difference,rule,totale_positivi,isolamento_domiciliare,totale_casi,dimessi_guariti,deceduti
0,2020-02-25T18:00:00,2020-02-25,150,114,35,149,1,Hospitalised,<NA>,<NA>,<NA>,<NA>,<NA>
1,2020-02-25T18:00:00,2020-02-25,150,<NA>,<NA>,312,-1,Active positives,311,162,<NA>,<NA>,<NA>
2,2021-11-07T17:00:00,2021-11-07,3613,<NA>,<NA>,96993,-6,Active positives,96987,93380,<NA>,<NA>,<NA>
3,2020-11-04T17:00:00,2020-11-04,<NA>,<NA>,<NA>,790360,17,Total cases,443235,<NA>,790377,307378,39747
4,2021-02-03T17:00:00,2021-02-03,<NA>,<NA>,<NA>,2583789,1,Total cases,434722,<NA>,2583790,2059248,89819
5,2021-06-23T17:00:00,2021-06-23,<NA>,<NA>,<NA>,4255416,18,Total cases,68619,<NA>,4255434,4059463,127334
6,2021-08-01T17:00:00,2021-08-01,<NA>,<NA>,<NA>,4355345,3,Total cases,91350,<NA>,4355348,4135930,128065
7,2021-08-11T17:00:00,2021-08-11,<NA>,<NA>,<NA>,4413160,2,Total cases,118761,<NA>,4413162,4166095,128304
8,2021-11-07T17:00:00,2021-11-07,<NA>,<NA>,<NA>,4808041,6,Total cases,96987,<NA>,4808047,4578669,132385
9,2022-01-11T17:00:00,2022-01-11,<NA>,<NA>,<NA>,7774636,227,Total cases,2134139,<NA>,7774863,5500938,139559


... (8 outputs left out)


Context around the mismatch: 2022-01-25T17:00:00  (difference = +24,537)


,data,totale_casi,nuovi_positivi
698,2022-01-22T17:00:00,9781191,171263
699,2022-01-23T17:00:00,9923678,138860
700,2022-01-24T17:00:00,10001344,77696
701,2022-01-25T17:00:00,10216395,190514
702,2022-01-26T17:00:00,10383561,167206
703,2022-01-27T17:00:00,10539601,155697
704,2022-01-28T17:00:00,10683948,143898



Change in active positives vs variazione_totale_positivi
   comparable: 1,780 | matching: 1,780 | mismatching: 0 | match: 100.0% | not comparable (first row of group / date gap / missing): 1
   no mismatches

=== National: cumulative columns must not decrease ===


,column,decreasing_steps,largest_drop,first_decrease,last_decrease
0,totale_casi,1,-148.0,2020-06-19,2020-06-19
1,deceduti,3,-40.0,2020-06-24,2024-02-23
2,dimessi_guariti,3,-618.0,2023-07-01,2024-06-13
3,tamponi,2,-27368510.0,2020-12-17,2024-11-05
4,casi_testati,7,-25196847.0,2020-12-06,2024-11-24
5,tamponi_test_molecolare,0,0.0,NaT,NaT
6,tamponi_test_antigenico_rapido,1,-27370138.0,2024-11-05,2024-11-05
7,totale_positivi_test_molecolare,9,-569200.0,2022-02-27,2024-09-26
8,totale_positivi_test_antigenico_rapido,6,-568809.0,2021-06-12,2024-09-25
9,casi_da_sospetto_diagnostico,17,-988470.0,2020-06-26,2020-12-03


In [17]:
nat_note_codes = profile_note_columns(df_national, "National")
print("\nDistinct note codes referenced by the national file:", len(nat_note_codes))


[National] note: non-empty rows=44 | unique tokens=53 | rows with several ';' codes=9 | code-like tokens=53 | other text tokens=0
   prefixes: {'dc': 35, 'pd': 11, 'nd': 7}

[National] note_test: non-empty rows=0 | unique tokens=0 | rows with several ';' codes=0 | code-like tokens=0 | other text tokens=0

[National] note_casi: non-empty rows=0 | unique tokens=0 | rows with several ';' codes=0 | code-like tokens=0 | other text tokens=0

Distinct note codes referenced by the national file: 53


## 5. Population dataset (`popolazione.csv`)

Checks: key uniqueness, numeric validity, `male + female = total`, identifier consistency, coordinates, **age-band coverage per region** and the presence of a reference year.

In [18]:
pop = df_population.copy()
POP_NUM = ["totale_genere_maschile", "totale_genere_femminile", "totale_generale"]
POP_KEYS = ["codice_regione", "range_eta"]

# ---- key uniqueness
complete_key = ~pop[POP_KEYS].apply(missing_mask).any(axis=1)
dup_key = pd.Series(False, index=pop.index)
dup_key.loc[complete_key] = pop.loc[complete_key].duplicated(subset=POP_KEYS, keep=False)
print("Rows:", len(pop), "| regions:", pop["codice_regione"].nunique(), "| age bands:", pop["range_eta"].nunique())
print("Rows with blank key:", int((~complete_key).sum()), "| duplicated (region, age band) keys:", int(dup_key.sum()))
if (~complete_key).any():
    add_issue("Population", "Keys", "Blank region code or age band", "High", int((~complete_key).sum()), "Key columns empty.", "Fix or drop.")
if dup_key.any():
    add_issue("Population", "Keys", "Duplicate (region, age band) rows", "High", int(dup_key.sum()), "Same key appears more than once.", "Aggregate or drop the duplicate.")
    save_exceptions("population_duplicate_keys", pop.loc[dup_key, POP_KEYS + POP_NUM])
    display(pop.loc[dup_key, POP_KEYS + POP_NUM])

# ---- numeric validity
rows = []
for c in POP_NUM:
    m = missing_mask(pop[c])
    raw = pop.loc[~m, c].astype("string").str.strip()
    num = pd.to_numeric(raw, errors="coerce")
    inv, neg = int(num.isna().sum()), int((num < 0).sum())
    nonint = int(((num.notna()) & (num % 1 != 0)).sum())
    rows.append({"column": c, "available": len(raw), "missing": int(m.sum()), "non_numeric": inv, "negative": neg,
                 "decimals": nonint, "min": num.min(), "max": num.max()})
    if inv:
        add_issue("Population", "Validity", f"Non-numeric values in {c}", "High", inv, "Cannot be cast to number.", "Fix before casting.")
    if neg:
        add_issue("Population", "Validity", f"Negative values in {c}", "High", neg, "Population cannot be negative.", "Investigate.")
    if nonint:
        add_issue("Population", "Validity", f"Decimal values in {c}", "Low", nonint, "Population counts should be integers.", "Round or investigate.")
    if int(m.sum()):
        add_issue("Population", "Missingness", f"Missing values in {c}", "High", int(m.sum()), "Population count missing.", "Source needed; do not impute with 0.")
display(pd.DataFrame(rows))

# ---- male + female = total
pop_num = pop.copy()
for c in POP_NUM:
    pop_num[c] = to_num(pop_num[c])
exp_total = pop_num["totale_genere_maschile"] + pop_num["totale_genere_femminile"]
diff_total = pop_num["totale_generale"] - exp_total
valid = pop_num["totale_generale"].notna() & exp_total.notna()
bad = valid & (diff_total != 0)
print(f"\nmale + female = total  -> checked {int(valid.sum())}, matching {int((valid & ~bad).sum())}, mismatching {int(bad.sum())}")
if bad.any():
    add_issue("Population", "Consistency", "male + female != total", "Medium", int(bad.sum()),
              f"Largest |difference| = {diff_total[bad].abs().max():,.0f}.", "Decide which figure is correct.")
    exc = pop_num.loc[bad, POP_KEYS + POP_NUM].assign(expected=exp_total[bad], difference=diff_total[bad])
    save_exceptions("population_male_female_total", exc)
    display(exc)

Rows: 210 | regions: 21 | age bands: 12
Rows with blank key: 0 | duplicated (region, age band) keys: 0


,column,available,missing,non_numeric,negative,decimals,min,max
0,totale_genere_maschile,210,0,0,0,0,457,805221
1,totale_genere_femminile,210,0,0,0,0,1307,802504
2,totale_generale,210,0,0,0,0,1764,1607725



male + female = total  -> checked 210, matching 210, mismatching 0


In [19]:
# ---- identifiers inside each region
id_cols = [c for c in ["denominazione_regione", "sigla_regione", "codice_nuts_1", "descrizione_nuts_1", "codice_nuts_2"] if c in pop.columns]
ident = pop.groupby("codice_regione", dropna=False)[id_cols].nunique(dropna=False)
bad_ident = ident[(ident != 1).any(axis=1)]
print("Regions with more than one value for an identifier column:", len(bad_ident))
if len(bad_ident):
    display(bad_ident)
    add_issue("Population", "Keys", "Identifier columns inconsistent within a region", "Medium", len(bad_ident),
              "A region has several names/NUTS/abbreviations.", "Build a clean region dimension.")

# ---- coordinates
if has_cols(pop, ["latitudine_regione", "longitudine_regione"]):
    geo = pop[["codice_regione", "latitudine_regione", "longitudine_regione"]].copy()
    geo["lat"], geo["lon"] = to_num(geo["latitudine_regione"]), to_num(geo["longitudine_regione"])
    bad_num = int(geo["lat"].isna().sum() + geo["lon"].isna().sum())
    per_reg = geo.groupby("codice_regione").agg(lat_unique=("lat", "nunique"), lon_unique=("lon", "nunique"), lat=("lat", "first"), lon=("lon", "first"))
    vary = per_reg[(per_reg["lat_unique"] > 1) | (per_reg["lon_unique"] > 1)]
    outside = per_reg[~per_reg["lat"].between(35.4, 47.2) | ~per_reg["lon"].between(6.5, 18.6)]
    print(f"Non-numeric/missing coordinates: {bad_num} | regions with varying coordinates: {len(vary)} | outside Italy bounding box: {len(outside)}")
    if bad_num:
        add_issue("Population", "Validity", "Missing/non-numeric coordinates", "Medium", bad_num, "lat/long cannot be parsed.", "Fix.")
    if len(vary):
        add_issue("Population", "Validity", "Coordinates vary within a region", "Medium", len(vary), "More than one lat/long per region.", "Keep one per region.")
    if len(outside):
        add_issue("Population", "Validity", "Coordinates outside Italy", "Medium", len(outside), f"Regions: {outside.index.tolist()}", "Verify.")

# ---- reference period
time_like = [c for c in pop.columns if re.search(r"anno|year|data|date|period", c, flags=re.I)]
print("\nColumns that could carry a reference year/date:", time_like or "none")
if not time_like:
    add_issue("Population", "Documentation", "No reference year/date in the population file", "Medium", len(pop),
              "The file has no column telling which year the population refers to.",
              "Ask the data owner / document the reference year; it matters for incidence rates per 100k.")

Regions with more than one value for an identifier column: 0
Non-numeric/missing coordinates: 0 | regions with varying coordinates: 0 | outside Italy bounding box: 0

Columns that could carry a reference year/date: none


In [20]:
# ---- age-band coverage per region
def parse_band(b):
    s = str(b).strip()
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.fullmatch(r"(\d+)\s*\+", s)
    if m:
        return int(m.group(1)), 999
    return None


band_rows = []
for code_, grp in pop.loc[~missing_mask(pop["range_eta"])].groupby("codice_regione"):
    bands = sorted(grp["range_eta"].astype("string").str.strip().unique().tolist(),
                   key=lambda b: (parse_band(b) or (9999, 9999)))
    parsed = [parse_band(b) for b in bands]
    unparseable = [b for b, p in zip(bands, parsed) if p is None]
    ok = [p for p in parsed if p is not None]
    problems = []
    for (lo1, hi1), (lo2, hi2) in zip(ok, ok[1:]):
        if lo2 != hi1 + 1:
            problems.append(f"{hi1}->{lo2}")
    band_rows.append({
        "codice_regione": code_,
        "denominazione_regione": grp["denominazione_regione"].iloc[0] if "denominazione_regione" in grp else "",
        "n_bands": len(bands),
        "youngest_age_covered": ok[0][0] if ok else None,
        "has_open_ended_band": any(hi == 999 for _, hi in ok),
        "gaps_or_overlaps": ", ".join(problems),
        "unparseable_bands": ", ".join(unparseable),
        "band_set": " | ".join(bands),
    })
bands_df = pd.DataFrame(band_rows)
standard_set = bands_df["band_set"].mode().iloc[0]
bands_df["same_as_most_common"] = bands_df["band_set"].eq(standard_set)
print("Most common band set:", standard_set)
print("Distinct age-band labels in the file:", pop["range_eta"].nunique())
display(bands_df.drop(columns=["band_set"]))

deviating = bands_df[~bands_df["same_as_most_common"]]
if len(deviating):
    names = deviating["denominazione_regione"].tolist()
    youngest = sorted(set(deviating["youngest_age_covered"].dropna().astype(int)))
    std_youngest = bands_df.loc[bands_df["same_as_most_common"], "youngest_age_covered"].dropna().astype(int).min()
    print("\nRegions using a different band set:", names)
    for _, r in deviating.iterrows():
        print(f"  {r['denominazione_regione']}: {r['band_set']}")
    add_issue("Population", "Consistency", "Different age-band classification between regions", "High", len(deviating),
              f"Regions {names} use a different band set from the most common one ({standard_set}). "
              f"Youngest age covered there: {youngest} vs {std_youngest} elsewhere, so age groups cannot be compared or summed directly.",
              "Define a harmonised age grouping (or restrict age analyses to the common ages); do not rename bands blindly.")
    save_exceptions("population_age_band_differences", deviating)
if (bands_df["gaps_or_overlaps"] != "").any():
    add_issue("Population", "Consistency", "Gaps/overlaps between consecutive age bands", "Medium",
              int((bands_df["gaps_or_overlaps"] != "").sum()), "See column gaps_or_overlaps in the age-band table.", "Verify band definitions.")
if (bands_df["unparseable_bands"] != "").any():
    add_issue("Population", "Validity", "Age-band labels that are not 'a-b' or 'a+'", "Medium",
              int((bands_df["unparseable_bands"] != "").sum()), "Labels cannot be parsed.", "Standardise labels.")
youngest_all = bands_df["youngest_age_covered"].dropna().astype(int)
if len(youngest_all) and youngest_all.min() > 0:
    add_issue("Population", "Completeness", f"No age band starting at 0 (youngest covered age = {youngest_all.min()})", "Medium", len(pop),
              "Children below that age are not in the population totals.",
              "Check the file's scope (e.g. vaccination target population vs total residents) before computing rates.")

# ---- totals
tot = pop_num.groupby("codice_regione").agg(total_generale=("totale_generale", "sum"),
                                            male=("totale_genere_maschile", "sum"),
                                            female=("totale_genere_femminile", "sum")).reset_index()
tot = tot.merge(bands_df[["codice_regione", "denominazione_regione", "youngest_age_covered"]], on="codice_regione", how="left")
print(f"\nSum of totale_generale over all rows: {tot['total_generale'].sum():,.0f}")
display(tot.sort_values("total_generale", ascending=False))
add_issue("Population", "Documentation", "Total population represented in the file", "Info", len(pop),
          f"Sum of totale_generale = {tot['total_generale'].sum():,.0f}.",
          "Compare with the expected scope (all residents ~59 million, or a target population) before using as denominator.")
save_csv(tot, "population_totals_by_region.csv")
save_csv(bands_df, "population_age_bands_by_region.csv")

Most common band set: 0-15 | 16-19 | 20-29 | 30-39 | 40-49 | 50-59 | 60-69 | 70-79 | 80-89 | 90+
Distinct age-band labels in the file: 12


,codice_regione,denominazione_regione,n_bands,youngest_age_covered,has_open_ended_band,gaps_or_overlaps,unparseable_bands,same_as_most_common
0,01,Piemonte,10,0,True,,,True
1,02,Valle d'Aosta,10,0,True,,,True
2,03,Lombardia,10,5,True,,,False
3,05,Veneto,10,0,True,,,True
4,06,Friuli Venezia Giulia,10,0,True,,,True
5,07,Liguria,10,0,True,,,True
6,08,Emilia-Romagna,10,0,True,,,True
7,09,Toscana,10,0,True,,,True
8,10,Umbria,10,0,True,,,True
9,11,Marche,10,0,True,,,True



Regions using a different band set: ['Lombardia']
  Lombardia: 05-11 | 12-19 | 20-29 | 30-39 | 40-49 | 50-59 | 60-69 | 70-79 | 80-89 | 90+

Sum of totale_generale over all rows: 59,210,972


,codice_regione,total_generale,male,female,denominazione_regione,youngest_age_covered
2,03,9597086,4698263,4898823,Lombardia,5
10,12,5755700,2779181,2976519,Lazio,0
13,15,5712143,2784616,2927527,Campania,0
3,05,4879133,2389717,2489416,Veneto,0
17,19,4875290,2370942,2504348,Sicilia,0
6,08,4464119,2173781,2290338,Emilia-Romagna,0
0,01,4311217,2095058,2216159,Piemonte,0
14,16,3953305,1923532,2029773,Puglia,0
7,09,3692555,1784318,1908237,Toscana,0
16,18,1894110,927732,966378,Calabria,0


## 6. Notes dataset (`note.csv`)

In [21]:
notes = df_notes.copy()
parsed = pd.to_datetime(notes["data"], errors="coerce")
if getattr(parsed.dt, "tz", None) is not None:
    parsed = parsed.dt.tz_localize(None)
notes["_day"] = parsed.dt.normalize()
print("Columns:", df_notes.columns.tolist())

# text column
if "note" in notes.columns:
    NOTE_TEXT = "note"
else:
    str_cols = [c for c in notes.columns if c not in ("data", "_day")]
    NOTE_TEXT = max(str_cols, key=lambda c: notes[c].astype("string").str.len().mean())
print("Text column used:", NOTE_TEXT)

# dates
inv = int(notes["_day"].isna().sum())
print(f"\nRows: {len(notes):,} | invalid dates: {inv} | range: {d(notes['_day'].min())} -> {d(notes['_day'].max())}")
print("Duplicate dates:", int(notes["_day"].duplicated(keep=False).sum()), "| sorted:", notes["_day"].is_monotonic_increasing)
if inv:
    add_issue("Notes", "Dates", "Unparseable dates", "High", inv, "Cannot be placed on the timeline.", "Fix or drop.")
dup_dates = notes["_day"].duplicated(keep=False) & notes["_day"].notna()
if dup_dates.any():
    add_issue("Notes", "Keys", "Several notes for the same date", "Low", int(dup_dates.sum()),
              "A date is not a unique key in the notes file.", "Aggregate notes per date or keep a note id.")
    save_exceptions("notes_duplicate_dates", notes.loc[dup_dates, ["data", NOTE_TEXT]])

# content
txt = notes[NOTE_TEXT].astype("string")
blank_ws = int((notes[NOTE_TEXT].notna() & txt.str.strip().eq("").fillna(False)).sum())
empty = missing_mask(notes[NOTE_TEXT])
non_empty = txt[~empty].str.strip()
dup_text = non_empty[non_empty.duplicated(keep=False)]
print(f"\nEmpty/whitespace notes: {int(empty.sum())} (whitespace-only strings: {blank_ws})")
print(f"Non-empty notes: {len(non_empty):,} | unique texts: {non_empty.nunique():,} | repeated texts: {int(non_empty.duplicated().sum())}")
lens = non_empty.str.len()
print("Length (characters): min", int(lens.min()), "| median", int(lens.median()), "| max", int(lens.max()))
if int(empty.sum()):
    add_issue("Notes", "Missingness", "Empty or whitespace-only notes", "Low", int(empty.sum()),
              "Date rows without any text.", "Drop or flag; they carry no information.")
    save_exceptions("notes_empty", notes.loc[empty, ["data", NOTE_TEXT]])
if len(dup_text):
    add_issue("Notes", "Duplicates", "Identical note text on different dates", "Low", int(non_empty.duplicated().sum()),
              "The same text is repeated.", "Keep (often a repeated disclaimer) or flag.")
    save_exceptions("notes_repeated_text", notes.loc[dup_text.index, ["data", NOTE_TEXT]].sort_values(NOTE_TEXT))
other_cols = [c for c in notes.columns if c not in ("data", "_day", NOTE_TEXT)]
if other_cols:
    print("Other columns in the notes file:", other_cols)

Columns: ['data', 'note']
Text column used: note

Rows: 1,617 | invalid dates: 0 | range: 2020-03-07 -> 2025-01-08
Duplicate dates: 0 | sorted: True

Empty/whitespace notes: 14 (whitespace-only strings: 14)
Non-empty notes: 1,603 | unique texts: 1,593 | repeated texts: 10
Length (characters): min 35 | median 789 | max 2325


In [22]:
# ---- date relationship with Regional and National
notes_days = set(notes["_day"].dropna())
reg_days = set(REG["df"]["_day"].dropna())
nat_days = set(NAT["df"]["_day"].dropna())
print(f"Notes days: {len(notes_days):,} | Regional days: {len(reg_days):,} | National days: {len(nat_days):,}")
print("Notes days NOT in regional:", len(notes_days - reg_days), "| NOT in national:", len(notes_days - nat_days))
reg_without_notes = sorted(reg_days - notes_days)
print("Regional days without a note:", len(reg_without_notes),
      f"({d(reg_without_notes[0])} -> {d(reg_without_notes[-1])})" if reg_without_notes else "")
if notes_days - reg_days:
    add_issue("Notes", "Keys", "Note dates that do not exist in the regional file", "Medium", len(notes_days - reg_days),
              "Orphan notes.", "Investigate before joining.")
if reg_without_notes:
    add_issue("Notes", "Completeness", "Regional days without any note", "Info", len(reg_without_notes),
              "Notes cover only part of the regional timeline.", "Left join and treat 'no note' explicitly; absence of a note is not an error.")
    save_exceptions("regional_days_without_notes", pd.DataFrame({"date": reg_without_notes}))

# ---- do the codes found in regional/national 'note' columns exist in the notes file?
all_codes = set(reg_note_codes) | set(nat_note_codes)
blob = "\n".join(notes[c].astype("string").fillna("").str.cat(sep="\n") for c in notes.columns if c != "_day")
found = sorted(c for c in all_codes if c in blob)
not_found = sorted(all_codes - set(found))
code_like_cols = [c for c in notes.columns if re.search(r"cod|id$", c, flags=re.I)]
print(f"\nCodes referenced by regional/national 'note': {len(all_codes)} | found anywhere in notes file: {len(found)} | not found: {len(not_found)}")
print("Notes-file columns that look like a code/id:", code_like_cols or "none")
if all_codes and not found:
    add_issue("Notes", "Keys", "Codes in regional/national 'note' cannot be resolved in the notes file", "Medium", len(all_codes),
              "None of the reference codes (e.g. 'xx-IT-###') appear in the notes file. The notes file joins by date at best.",
              "Do not join by code. Get the code catalogue from the source or treat 'note' codes as opaque flags.")
elif not_found:
    add_issue("Notes", "Keys", "Some referenced note codes missing from the notes file", "Low", len(not_found),
              f"Examples: {not_found[:5]}", "Verify with the source.")
    save_exceptions("note_codes_not_found", pd.DataFrame({"code": not_found}))

# ---- region mentions in the free text (exploratory)
def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().replace("\u2019", "'").replace("`", "'").strip()


ALIASES = {"p.a. bolzano": ["bolzano", "alto adige", "sudtirol"], "p.a. trento": ["trento", "trentino"],
           "valle d'aosta": ["aosta"], "friuli venezia giulia": ["friuli"]}
region_names = reg_dim["denominazione_regione"].dropna().astype(str).str.strip().unique().tolist()
patterns = {r: list({norm(r), *ALIASES.get(norm(r), [])}) for r in region_names}
mention_counts = {r: 0 for r in region_names}
regions_per_note = []
for t in non_empty:
    n = norm(t)
    hit = [r for r, pats in patterns.items() if any(p in n for p in pats)]
    for r in hit:
        mention_counts[r] += 1
    regions_per_note.append(len(hit))
rpn = pd.Series(regions_per_note, dtype="int64")
print(f"\nNotes mentioning at least one region: {int((rpn > 0).sum())} of {len(rpn)} ({pct(int((rpn > 0).sum()), len(rpn))}%) "
      f"| mentioning 3+ regions: {int((rpn >= 3).sum())}")
display(pd.Series(mention_counts, name="notes_mentioning").sort_values(ascending=False).to_frame())
add_issue("Notes", "Relationship", "Notes are free text; no key to a single region or record", "Info", len(non_empty),
          f"{pct(int((rpn > 0).sum()), len(rpn))}% of notes mention a region name, but many mention several; matching is exploratory only.",
          "Keep notes as a separate date-level table (or NLP-derived flags); do not merge into regional rows.")

Notes days: 1,617 | Regional days: 1,781 | National days: 1,781
Notes days NOT in regional: 0 | NOT in national: 0
Regional days without a note: 164 (2020-02-24 -> 2021-04-05)

Codes referenced by regional/national 'note': 106 | found anywhere in notes file: 0 | not found: 106
Notes-file columns that look like a code/id: none

Notes mentioning at least one region: 1542 of 1603 (96.19%) | mentioning 3+ regions: 1235


,notes_mentioning
Umbria,1036
Basilicata,833
Friuli Venezia Giulia,664
Emilia-Romagna,658
Sicilia,552
Abruzzo,540
Campania,516
Veneto,397
P.A. Bolzano,281
Liguria,268


## 7. Cross-dataset checks

These are the checks that make joins and features safe: do the 21 regions add up to the national figure, and do the regional and population files describe the same regions?

In [23]:
# ---- 7.1 sum of regions vs national, per day and per column
reg_num, nat_num = REG["num"], NAT["num"]
add_cols = [c for c in NUM_COLS if c in reg_num.columns and c in nat_num.columns]
n_regions = reg_num["codice_regione"].nunique()

reg_sum = reg_num.groupby("_day")[add_cols].sum(min_count=1)
reg_cnt = reg_num.groupby("_day")[add_cols].count()
nat_idx = nat_num.dropna(subset=["_day"]).drop_duplicates("_day").set_index("_day")[add_cols]
common_days = reg_sum.index.intersection(nat_idx.index)
print(f"Regions: {n_regions} | days only in regional: {len(reg_sum.index.difference(nat_idx.index))} | days only in national: {len(nat_idx.index.difference(reg_sum.index))}")
if len(reg_sum.index.difference(nat_idx.index)) or len(nat_idx.index.difference(reg_sum.index)):
    add_issue("Cross-dataset", "Keys", "Regional and national files cover different days", "High",
              len(reg_sum.index.symmetric_difference(nat_idx.index)), "Day sets differ.", "Align the calendars before comparing or joining.")

rows, long_frames = [], []
for c in add_cols:
    comparable = (reg_cnt.loc[common_days, c] == n_regions) & nat_idx.loc[common_days, c].notna()
    days_c = comparable[comparable].index
    diff = reg_sum.loc[days_c, c] - nat_idx.loc[days_c, c]
    bad = diff[diff != 0]
    rows.append({"column": c, "days_compared": len(days_c), "days_excluded_partial_or_missing": int(len(common_days) - len(days_c)),
                 "days_mismatching": len(bad), "mismatch_pct": pct(len(bad), len(days_c)),
                 "max_abs_difference": float(bad.abs().max()) if len(bad) else 0,
                 "first_mismatch": bad.index.min() if len(bad) else pd.NaT,
                 "last_mismatch": bad.index.max() if len(bad) else pd.NaT})
    if len(bad):
        long_frames.append(pd.DataFrame({"date": bad.index, "column": c, "sum_of_regions": reg_sum.loc[bad.index, c].to_numpy(),
                                         "national": nat_idx.loc[bad.index, c].to_numpy(), "difference": bad.to_numpy()}))
        add_issue("Cross-dataset", "Reconciliation", f"Sum of regions != national for {c}", "High" if pct(len(bad), len(days_c)) > 5 else "Medium",
                  len(bad), f"{len(bad)} of {len(days_c)} comparable days differ ({pct(len(bad), len(days_c))}%); max |difference| = {bad.abs().max():,.0f}; "
                  f"{d(bad.index.min())} -> {d(bad.index.max())}.",
                  "Decide the source of truth (regional sums vs national) for each variable; national may include corrections not distributed to regions.")
recon = pd.DataFrame(rows)
display(recon)
save_csv(recon, "reconciliation_regional_vs_national.csv")
if long_frames:
    save_exceptions("cross_regional_sum_vs_national", pd.concat(long_frames, ignore_index=True))

Regions: 21 | days only in regional: 0 | days only in national: 0


,column,days_compared,days_excluded_partial_or_missing,days_mismatching,mismatch_pct,max_abs_difference,first_mismatch,last_mismatch
0,ricoverati_con_sintomi,1781,0,0,0.00,0.0,NaT,NaT
1,terapia_intensiva,1781,0,0,0.00,0.0,NaT,NaT
2,totale_ospedalizzati,1781,0,1,0.06,1.0,2020-02-25,2020-02-25
3,isolamento_domiciliare,1781,0,0,0.00,0.0,NaT,NaT
4,totale_positivi,1781,0,0,0.00,0.0,NaT,NaT
5,variazione_totale_positivi,1781,0,0,0.00,0.0,NaT,NaT
6,nuovi_positivi,1781,0,0,0.00,0.0,NaT,NaT
7,dimessi_guariti,1781,0,0,0.00,0.0,NaT,NaT
8,deceduti,1781,0,0,0.00,0.0,NaT,NaT
9,casi_da_sospetto_diagnostico,162,1619,0,0.00,0.0,NaT,NaT


In [24]:
# ---- 7.2 regional vs population: keys, names, coordinates, NUTS
def norm_code(s):
    return s.astype("string").str.strip().str.lstrip("0")


reg_ids = reg_dim.copy()
reg_ids["code_norm"] = norm_code(reg_ids["codice_regione"])
pop_ids_cols = ["codice_regione"] + [c for c in ["denominazione_regione", "codice_nuts_1", "codice_nuts_2"] if c in pop.columns]
pop_ids = pop[pop_ids_cols].drop_duplicates(subset=["codice_regione"]).copy()
pop_ids["code_norm"] = norm_code(pop_ids["codice_regione"])

raw_equal = set(reg_ids["codice_regione"].astype(str)) == set(pop_ids["codice_regione"].astype(str))
norm_equal = set(reg_ids["code_norm"].dropna()) == set(pop_ids["code_norm"].dropna())
print("Region code sets identical as text:", raw_equal, "| identical after removing leading zeros:", norm_equal)
print("Regional codes not in population:", sorted(set(reg_ids["code_norm"].dropna()) - set(pop_ids["code_norm"].dropna())))
print("Population codes not in regional:", sorted(set(pop_ids["code_norm"].dropna()) - set(reg_ids["code_norm"].dropna())))
if not norm_equal:
    add_issue("Cross-dataset", "Keys", "Region codes differ between regional and population files", "High",
              len(set(reg_ids["code_norm"].dropna()) ^ set(pop_ids["code_norm"].dropna())), "Code sets do not match.", "Build a mapping table before joining.")
elif not raw_equal:
    add_issue("Cross-dataset", "Keys", "Region codes formatted differently (e.g. '3' vs '03')", "Medium", len(reg_ids),
              "Codes match only after removing leading zeros.", "Cast both to the same string format before joining.")

merged = reg_ids.merge(pop_ids, on="code_norm", how="outer", suffixes=("_regional", "_population"), indicator=True)
if "denominazione_regione_population" in merged.columns:
    merged["names_equal"] = merged["denominazione_regione_regional"].map(norm) == merged["denominazione_regione_population"].map(norm)
    name_diff = merged[(merged["_merge"] == "both") & ~merged["names_equal"]]
    print("\nRegions whose NAME differs between files:", len(name_diff))
    if len(name_diff):
        display(name_diff[["code_norm", "denominazione_regione_regional", "denominazione_regione_population"]])
        add_issue("Cross-dataset", "Keys", "Region names differ between regional and population files", "Medium", len(name_diff),
                  "Same code, different spelling.", "Join on code, not on name; keep one canonical name.")
        save_exceptions("cross_region_name_differences", name_diff)

# coordinates
if has_cols(df_regional, ["lat", "long"]) and has_cols(pop, ["latitudine_regione", "longitudine_regione"]):
    g1 = per_region.reset_index()[["codice_regione", "lat", "long"]]
    g1["code_norm"] = norm_code(g1["codice_regione"])
    g2 = pop.groupby("codice_regione").agg(plat=("latitudine_regione", lambda s: to_num(s).iloc[0]),
                                           plong=("longitudine_regione", lambda s: to_num(s).iloc[0])).reset_index()
    g2["code_norm"] = norm_code(g2["codice_regione"])
    gg = g1.merge(g2, on="code_norm")
    gg["lat_diff"] = (gg["lat"] - gg["plat"]).abs()
    gg["long_diff"] = (gg["long"] - gg["plong"]).abs()
    far = gg[(gg["lat_diff"] > 1e-3) | (gg["long_diff"] > 1e-3)]
    print(f"\nRegions where regional and population coordinates differ (> 0.001 deg): {len(far)}")
    if len(far):
        display(far[["code_norm", "lat", "plat", "long", "plong"]])
        add_issue("Cross-dataset", "Validity", "Coordinates differ between regional and population files", "Low", len(far),
                  "Same region, different lat/long.", "Pick one source for coordinates.")

# NUTS
for c in ["codice_nuts_1", "codice_nuts_2"]:
    if c in df_regional.columns and c in pop.columns:
        r_av = reg_df.loc[~missing_mask(reg_df[c]), ["codice_regione", c]].drop_duplicates("codice_regione")
        r_av["code_norm"] = norm_code(r_av["codice_regione"])
        p_av = pop_ids[["code_norm", c]].rename(columns={c: c + "_pop"})
        cmp_ = r_av.merge(p_av, on="code_norm", how="inner")
        conflict = cmp_[cmp_[c].astype("string").str.strip() != cmp_[c + "_pop"].astype("string").str.strip()]
        recoverable = int(missing_mask(reg_df[c]).sum())
        print(f"{c}: regional rows empty = {recoverable:,} (recoverable from the population file via region code); conflicting values = {len(conflict)}")
        if len(conflict):
            add_issue("Cross-dataset", "Keys", f"{c} conflicts between regional and population", "Medium", len(conflict), "Different NUTS value for the same region.", "Verify.")
        elif recoverable:
            add_issue("Cross-dataset", "Missingness", f"{c} empty in regional rows but available in population", "Info", recoverable,
                      "Values are constant per region and can be restored from the population file.", "Fill from the region dimension instead of treating as missing data.")

Region code sets identical as text: True | identical after removing leading zeros: True
Regional codes not in population: []
Population codes not in regional: []

Regions whose NAME differs between files: 2


,code_norm,denominazione_regione_regional,denominazione_regione_population
13,21,P.A. Bolzano,Bolzano
14,22,P.A. Trento,Trento



Regions where regional and population coordinates differ (> 0.001 deg): 0
codice_nuts_1: regional rows empty = 6,846 (recoverable from the population file via region code); conflicting values = 0
codice_nuts_2: regional rows empty = 6,846 (recoverable from the population file via region code); conflicting values = 0


## 8. Issue Register and hand-off to Data Cleaning

Every finding above is collected in one table, sorted by severity. Row-level evidence is exported to `outputs/data_quality/exceptions/`.

In [25]:
register = pd.DataFrame(ISSUES)
register["severity_rank"] = register["severity"].map(SEVERITY_ORDER)
register = (register.sort_values(["severity_rank", "dataset", "area", "check"])
            .drop(columns="severity_rank").reset_index(drop=True))
register.insert(0, "issue_id", [f"DQ-{i + 1:03d}" for i in range(len(register))])

print(f"Total findings: {len(register)}\n")
display(pd.crosstab(register["dataset"], register["severity"]).reindex(columns=[s for s in SEVERITY_ORDER if s in register["severity"].unique()]))

pd.set_option("display.max_colwidth", 200)
display(register)
save_csv(register, "data_quality_issue_register.csv")

# row-level evidence
EXC_DIR = OUTPUT_DIR / "exceptions"
EXC_DIR.mkdir(exist_ok=True)
exc_index = []
for name, frame in EXCEPTIONS.items():
    safe = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    frame.to_csv(EXC_DIR / f"{safe}.csv", index=False, encoding="utf-8-sig")
    exc_index.append({"file": f"exceptions/{safe}.csv", "rows": len(frame), "columns": ", ".join(frame.columns[:8]) + (" ..." if frame.shape[1] > 8 else "")})
exc_index = pd.DataFrame(exc_index)
print(f"\nException files written: {len(exc_index)}")
if len(exc_index):
    display(exc_index)
    save_csv(exc_index, "exceptions_index.csv")

print("\nFiles in", OUTPUT_DIR)
for f in sorted(set(SAVED_FILES)):
    print("  ", f)
print("\nThe source DataFrames were not modified:",
      all(frame is DATASETS[n] for n, frame in zip(["Regional", "National", "Population", "Notes"], [df_regional, df_national, df_population, df_notes])))

Total findings: 78



severity,High,Medium,Low,Info
dataset,,,,
Cross-dataset,0,2,0,2
National,3,24,2,0
Notes,0,1,3,2
Population,1,1,0,1
Regional,1,30,5,0


,issue_id,dataset,area,check,severity,affected,detail,suggested_cleaning_action
0,DQ-001,National,Consistency,Daily new cases vs change in totale_casi,High,1033,"1033 of 1780 comparable rows mismatch (58.03%): 758 negative, 275 positive. 578 of them are within +/-5, 224 exceed 100. Largest |difference| = 46,005 on 2022-07-11. Cause not established by this ...",Do not trust 'nuovi_positivi' as the daily change of 'totale_casi' without reconciling; decide which of the two to use as the target series.
1,DQ-002,National,Missingness,Missing values in note_casi,High,1781,Column is empty in every row.,Drop from model tables or document why it is empty.
2,DQ-003,National,Missingness,Missing values in note_test,High,1781,Column is empty in every row.,Drop from model tables or document why it is empty.
3,DQ-004,Population,Consistency,Different age-band classification between regions,High,1,Regions ['Lombardia'] use a different band set from the most common one (0-15 | 16-19 | 20-29 | 30-39 | 40-49 | 50-59 | 60-69 | 70-79 | 80-89 | 90+). Youngest age covered there: [5] vs 0 elsewhere...,Define a harmonised age grouping (or restrict age analyses to the common ages); do not rename bands blindly.
4,DQ-005,Regional,Consistency,Daily new cases vs change in totale_casi,High,1923,"1923 of 37380 comparable rows mismatch (5.14%): 1623 negative, 300 positive. 1379 of them are within +/-5, 231 exceed 100. Largest |difference| = 46,005 on 2022-07-11. Cause not established by thi...",Do not trust 'nuovi_positivi' as the daily change of 'totale_casi' without reconciling; decide which of the two to use as the target series.
5,DQ-006,Cross-dataset,Keys,Region names differ between regional and population files,Medium,2,"Same code, different spelling.","Join on code, not on name; keep one canonical name."
6,DQ-007,Cross-dataset,Reconciliation,Sum of regions != national for totale_ospedalizzati,Medium,1,1 of 1781 comparable days differ (0.06%); max |difference| = 1; 2020-02-25 -> 2020-02-25.,Decide the source of truth (regional sums vs national) for each variable; national may include corrections not distributed to regions.
7,DQ-008,National,Consistency,Active positives: totale_positivi = totale_ospedalizzati + isolamento_domiciliare,Medium,2,2 of 1781 rows break the rule (0.11%). Dates 2020-02-25 -> 2021-11-07; largest |difference| = 6 on 2021-11-07.,Keep raw values; decide which component is wrong (see linked mismatches) and whether to recompute the total.
8,DQ-009,National,Consistency,Cumulative column casi_da_screening decreases,Medium,1,"1 decreases (largest drop -653,140); 2020-12-03 -> 2020-12-03.",Usually a source correction. Do not take diff() of this column as daily counts without handling negatives.
9,DQ-010,National,Consistency,Cumulative column casi_da_sospetto_diagnostico decreases,Medium,17,"17 decreases (largest drop -988,470); 2020-06-26 -> 2020-12-03.",Usually a source correction. Do not take diff() of this column as daily counts without handling negatives.



Exception files written: 18


,file,rows,columns
0,exceptions/regional_numeric_problems.csv,19,"data, codice_regione, denominazione_regione, column, value, problem"
1,exceptions/regional_rule_active_positives.csv,1,"data, codice_regione, denominazione_regione, date_parsed, totale_positivi, totale_ospedalizzati, isolamento_domiciliare, expected ..."
2,exceptions/regional_rule_total_cases.csv,15,"data, codice_regione, denominazione_regione, date_parsed, totale_casi, totale_positivi, dimessi_guariti, deceduti ..."
3,exceptions/regional_linked_rule_mismatches.csv,1,"data, codice_regione, denominazione_regione, rules, n_rules"
4,exceptions/regional_daily_vs_cumulative_nuovi_positivi.csv,1923,"data, codice_regione, denominazione_regione, date_parsed, previous_date, gap_days, totale_casi, daily_change_in_totale_casi ..."
5,exceptions/regional_decreasing_cumulative.csv,1414,"column, data, codice_regione, denominazione_regione, date_parsed, totale_casi, change_vs_previous_value, deceduti ..."
6,exceptions/national_rule_hospitalised.csv,1,"data, date_parsed, totale_ospedalizzati, ricoverati_con_sintomi, terapia_intensiva, expected, difference"
7,exceptions/national_rule_active_positives.csv,2,"data, date_parsed, totale_positivi, totale_ospedalizzati, isolamento_domiciliare, expected, difference"
8,exceptions/national_rule_total_cases.csv,13,"data, date_parsed, totale_casi, totale_positivi, dimessi_guariti, deceduti, expected, difference"
9,exceptions/national_linked_rule_mismatches.csv,2,"data, rules, n_rules"



Files in C:\Users\New Star Computer 25\EpiShield-AI\Src\Data Engineering\outputs\data_quality
   arithmetic_rules_national.csv
   arithmetic_rules_regional.csv
   column_profile_national.csv
   column_profile_notes.csv
   column_profile_population.csv
   column_profile_regional.csv
   data_quality_issue_register.csv
   exceptions_index.csv
   missingness_national.csv
   missingness_regional.csv
   monotonic_check_national.csv
   monotonic_check_regional.csv
   numeric_validation_national.csv
   numeric_validation_regional.csv
   population_age_bands_by_region.csv
   population_totals_by_region.csv
   quality_summary.csv
   reconciliation_regional_vs_national.csv

The source DataFrames were not modified: True


### Hand-off rules for the Cleaning phase

1. **Start from the Issue Register** (`data_quality_issue_register.csv`). Fix issues in severity order; every cleaning step should reference an `issue_id`.
2. **Keep the raw layer untouched** and write cleaned tables to a separate location.
3. **Missing is not zero.** Testing columns and diagnostic-case columns are *not reported* before their first available date. Use an availability window or a flag instead of filling with 0.
4. **Cumulative vs daily columns disagree** (see the day-to-day rules). Decide which series is the modelling target, and never take `diff()` of a cumulative column without handling decreases.
5. **Regional sums vs national** differ for some variables. Decide the source of truth per variable.
6. **Population denominators:** harmonise age bands (regions with a different band set) and confirm the reference year/scope before computing rates.
7. **Notes** are free text at date level. Keep them as a separate table; do not merge into regional rows.
8. Use the files in `exceptions/` as evidence: they list exactly which rows triggered each finding.